In [1]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from kdutils.macro2 import *

In [3]:
method = 'ricso2'
instruments = 'rbb'
task_id = '113001'
period = '5'
model_id = '1018806311332385'

In [5]:
base_path1 = os.path.join(base_path, method, instruments, 'temp', 'model',
                              str(task_id), str(period), 'rl')

dirs_path1 = os.path.join(base_path1, "composite", "model", 'rl',
                             str(model_id), 'data')

dirs_path2 = os.path.join(base_path1, "signal", 'rl', str(model_id))

In [6]:
test_data = pd.read_feather(os.path.join(dirs_path1,
                                             "wf_test_data.feather"))
test_data1 = pd.read_feather(
        os.path.join(base_path1, "data", "{0}_data.feather".format('test')))

test_data = test_data.drop(['signal'], axis=1).merge(
        test_data1[['trade_time', 'code', 'nxt1_ret_5h']],
        on=['trade_time', 'code'])
test_data.head()

,trade_time,symbol,code,task_id,value,confidence,net_er_out,er_value,nxt1_ret_5h
0,2024-04-18 21:04:00,rbb9999,RB,113001,"[-0.9177427291870117, 0.2626307010650635, -0.4...",0.881753,0.881753,0.881753,0.000946
1,2024-04-18 21:05:00,rbb9999,RB,113001,"[-0.7601301670074463, 0.7531843185424805, 0.29...",0.723062,0.723062,0.723062,0.000422
2,2024-04-18 21:06:00,rbb9999,RB,113001,"[-0.21360868215560913, 0.16371333599090576, -0...",0.735966,0.735966,0.735966,0.000629
3,2024-04-18 21:07:00,rbb9999,RB,113001,"[-0.7692319750785828, 0.7072559595108032, -0.2...",0.958070,0.958070,0.958070,0.000352
4,2024-04-18 21:08:00,rbb9999,RB,113001,"[-0.6830067038536072, -0.211450457572937, -0.1...",0.197368,-0.197368,-0.197368,0.000031


#### er 分桶单调性
- 目的: 判断 er 整体是否像 alpha 因子。
- 做法: 把 er 从小到大分成 10 桶； 每桶计算未来收益均值。
- 结果: 低 er 桶未来收益偏负； 高 er 桶未来收益偏正；中间桶接近 0； 整体从低到高单调上升。
- 说明: er 是一个有效连续 alpha。保留 er 的连续强度，而不是硬转离散信号。
- 判断标准：ret_mean 是否随 er_mean 单调上升。signed_pnl_mean 是否大部分为正。

In [7]:
er_col = 'net_er_out'
ret_col = 'nxt1_ret_5h'
bins = 10

In [8]:
x = test_data[[er_col, ret_col]].replace([np.inf, -np.inf], np.nan).dropna()
x["bucket"] = pd.qcut(x[er_col], q=bins, duplicates="drop")
x["signed_pnl"] = x[er_col] * x[ret_col]
res = x.groupby("bucket").agg(
        count=(er_col, "size"),
        er_mean=(er_col, "mean"),
        ret_mean=(ret_col, "mean"),
        signed_pnl_mean=("signed_pnl", "mean"),
        win_rate=("signed_pnl", lambda s: (s > 0).mean()),
    )
res

,count,er_mean,ret_mean,signed_pnl_mean,win_rate
bucket,,,,,
"(-1.0, -0.831]",16962,-0.905433,-2.839426e-05,2.687733e-05,0.535196
"(-0.831, -0.689]",16962,-0.761380,-3.166665e-05,2.388648e-05,0.532366
"(-0.689, -0.38]",16962,-0.559107,-1.718048e-05,9.730012e-06,0.520988
"(-0.38, 0.0]",28402,-0.056566,-7.672277e-06,7.965776e-07,0.134709
"(0.0, 0.289]",5521,0.157567,5.118440e-06,-1.989326e-06,0.506792
"(0.289, 0.668]",16962,0.518423,1.502782e-05,7.971278e-06,0.499941
"(0.668, 0.794]",16961,0.738789,1.128223e-05,8.362039e-06,0.503331
"(0.794, 0.861]",16962,0.829432,-1.073350e-05,-8.880987e-06,0.499293
"(0.861, 0.925]",16962,0.891570,8.612513e-07,8.680938e-07,0.502771


#### abs(er) 强度有效性
- 目的: 判断 er 的强度是否值得保留。
- 做法: 按 abs(er) 从小到大分桶；每桶计算 sign(er) * future_return。
- 结果: abs(er) 越大，sign(er) * future_return 越高
- 说明: 强度有效。
- 判断标准: directional_ret_mean 是否随着 abs_er_mean 增大而提高。

In [9]:
x = test_data[[er_col, ret_col]].replace([np.inf, -np.inf], np.nan).dropna()
x["abs_er"] = x[er_col].abs()
x["direction"] = np.sign(x[er_col])
x["directional_ret"] = x["direction"] * x[ret_col]
x["bucket"] = pd.qcut(x["abs_er"], q=bins, duplicates="drop")
res = x.groupby("bucket").agg(
        count=("abs_er", "size"),
        abs_er_mean=("abs_er", "mean"),
        directional_ret_mean=("directional_ret", "mean"),
        directional_ret_sum=("directional_ret", "sum"),
        direction_win_rate=("directional_ret", lambda s: (s > 0).mean()),
    )
res

,count,abs_er_mean,directional_ret_mean,directional_ret_sum,direction_win_rate
bucket,,,,,
"(-0.001, 0.337]",33924,0.071345,0.000004,0.132682,0.194435
"(0.337, 0.554]",16962,0.457487,0.000019,0.320408,0.512086
"(0.554, 0.679]",16961,0.622728,0.000015,0.252028,0.507989
"(0.679, 0.752]",16962,0.717228,0.000016,0.275100,0.513560
"(0.752, 0.808]",16962,0.780578,0.000025,0.429954,0.520693
"(0.808, 0.852]",16961,0.830182,-0.000006,-0.107225,0.506043
"(0.852, 0.895]",16962,0.873057,-0.000004,-0.064288,0.511732
"(0.895, 0.943]",16962,0.917871,0.000022,0.377992,0.513029
"(0.943, 0.999]",16962,0.968650,0.000009,0.147999,0.517451


In [12]:
### 生成信号的分析
signal_data = pd.read_feather(os.path.join(dirs_path2, "erband_discrete_signal", "1002_test.feather"))
signal_data.head()

,trade_time,code,signal,symbol,task_id,value,confidence,net_er_out,er_value,nxt1_ret_5h
0,2024-04-18 21:04:00,RB,1,rbb9999,113001,"[-0.9177427291870117, 0.2626307010650635, -0.4...",0.881753,0.881753,0.881753,0.000946
1,2024-04-18 21:05:00,RB,1,rbb9999,113001,"[-0.7601301670074463, 0.7531843185424805, 0.29...",0.723062,0.723062,0.723062,0.000422
2,2024-04-18 21:06:00,RB,1,rbb9999,113001,"[-0.21360868215560913, 0.16371333599090576, -0...",0.735966,0.735966,0.735966,0.000629
3,2024-04-18 21:07:00,RB,1,rbb9999,113001,"[-0.7692319750785828, 0.7072559595108032, -0.2...",0.958070,0.958070,0.958070,0.000352
4,2024-04-18 21:08:00,RB,0,rbb9999,113001,"[-0.6830067038536072, -0.211450457572937, -0.1...",0.197368,-0.197368,-0.197368,0.000031


In [16]:
signal_col = 'signal'
ret_col = 'nxt1_ret_5h'
value_col = 'net_er_out'
df = signal_data[[signal_col, ret_col, value_col]].replace(
        [np.inf, -np.inf], np.nan
    ).dropna()
# 1. 信号分布
signal_dist = df[signal_col].value_counts(dropna=False).sort_index()
signal_ratio = df[signal_col].value_counts(
        normalize=True, dropna=False
    ).sort_index()

# 2. 按 signal 分组看未来收益
by_signal = df.groupby(signal_col).agg(
        count=(ret_col, "size"),
        ratio=(ret_col, lambda s: len(s) / len(df)),
        value_mean=(value_col, "mean"),
        abs_value_mean=(value_col, lambda s: s.abs().mean()),
        ret_mean=(ret_col, "mean"),
        ret_sum=(ret_col, "sum"),
        ret_std=(ret_col, "std"),
        win_rate=(ret_col, lambda s: (s > 0).mean()),
    )

# 3. 按交易方向计算 directional return
# signal=1 希望 ret > 0
# signal=-1 希望 ret < 0
# signal=0 希望 ret 接近 0
df["directional_ret"] = df[signal_col] * df[ret_col]
active = df[signal_col] != 0
active_stats = {
        "active_ratio": active.mean(),
        "active_count": int(active.sum()),
        "active_directional_ret_mean": df.loc[active, "directional_ret"].mean(),
        "active_directional_ret_sum": df.loc[active, "directional_ret"].sum(),
        "active_win_rate": (df.loc[active, "directional_ret"] > 0).mean(),
        "flat_ret_mean": df.loc[df[signal_col] == 0, ret_col].mean(),
        "flat_abs_ret_mean": df.loc[df[signal_col] == 0, ret_col].abs().mean(),
    }

In [17]:
signal_dist

-1    51787
 0    34625
 1    83206
Name: signal, dtype: int64

In [18]:
signal_ratio

-1    0.305315
 0    0.204135
 1    0.490549
Name: signal, dtype: float64

In [19]:
by_signal

,count,ratio,value_mean,abs_value_mean,ret_mean,ret_sum,ret_std,win_rate
signal,,,,,,,,
-1,51787,0.305315,-0.735417,0.735417,-0.000026,-1.331528,0.001239,0.470639
0,34625,0.204135,0.003058,0.076856,-0.000006,-0.207912,0.001224,0.495249
1,83206,0.490549,0.796704,0.796704,0.000004,0.320058,0.001169,0.502860


In [20]:
active_stats

{'active_ratio': 0.7958648256670873,
 'active_count': 134993,
 'active_directional_ret_mean': 1.223460511323769e-05,
 'active_directional_ret_sum': 1.6515860480512954,
 'active_win_rate': 0.5129377078811493,
 'flat_ret_mean': -6.00466457094032e-06,
 'flat_abs_ret_mean': 0.0007380635050614516}